# Milestone 4: SME Early Warning Watchlist

## Objective

Rank historically booked SME accounts for review using their repayment,
utilization, deposit, and overdraft behaviour.

## Data and Timeline

- Population: 9,887 historically booked accounts.
- Observation window: 24 months, indexed from 0 to 23.
- Target: `deterioration_next_6_12mo`.

The target was generated by the Milestone 1 add-on script using a
probability formula and a random draw. It represents simulated future
deterioration; it is not an observed outcome from additional future months.

We will summarize behaviour into one row per account, train and evaluate
a model on separate accounts, and produce a watchlist with warning reasons.

## 1. Load and Validate the Saved Data

Read the existing booked-account and monthly-panel files.
This notebook does not regenerate or overwrite the source data.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Find the project root from either the root or notebooks directory.
current = Path.cwd()
root = next(
    (
        folder for folder in [current, *current.parents]
        if (folder / "data/raw/sme_onbook_portfolio.csv").is_file()
    ),
    None,
)

if root is None:
    raise FileNotFoundError("Could not locate data/raw/sme_onbook_portfolio.csv")

raw_directory = root / "data" / "raw"

onbook = pd.read_csv(
    raw_directory / "sme_onbook_portfolio.csv",
    dtype={"applicant_id": "string"},
)
panel = pd.read_csv(
    raw_directory / "sme_behavioural_panel.csv",
    dtype={"applicant_id": "string"},
)

TARGET = "deterioration_next_6_12mo"

required_panel_columns = [
    "applicant_id", "month_index", "credit_limit",
    "outstanding_balance", "utilization_onbook",
    "days_past_due", "deposit_inflow", "overdraft_count",
]

assert set(required_panel_columns).issubset(panel.columns)
assert {"applicant_id", TARGET}.issubset(onbook.columns)

assert onbook["applicant_id"].notna().all()
assert onbook["applicant_id"].is_unique
assert onbook[TARGET].isin([0, 1]).all()
assert onbook[TARGET].nunique() == 2

assert panel[required_panel_columns].notna().all().all()
assert not panel.duplicated(["applicant_id", "month_index"]).any()
assert set(panel["applicant_id"]) == set(onbook["applicant_id"])
assert panel["month_index"].isin(range(24)).all()
assert panel.groupby("applicant_id")["month_index"].nunique().eq(24).all()

print(f"Booked accounts: {len(onbook):,}")
print(f"Monthly records: {len(panel):,}")
print(f"Simulated deterioration cases: {onbook[TARGET].sum():,}")
print(f"Simulated deterioration rate: {onbook[TARGET].mean():.2%}")
print("Input checks passed.")

Booked accounts: 9,887
Monthly records: 237,288
Simulated deterioration cases: 1,959
Simulated deterioration rate: 19.81%
Input checks passed.


## 2. Create Account-Level Behavioural Features

Summarize the 24-month observation window into one row per account.

- Recent behaviour: months 22–24.
- Trend comparisons: months 19–24 versus months 1–6.
- Delinquency history: maximum DPD and number of delinquent months.
- Overdraft history: total and recent overdraft counts.

A positive utilization change means utilization increased.
A positive deposit decline means average deposit inflow decreased.

Add application-time leverage, DSCR, and bureau score as historical
background information. These are not refreshed month-24 financials.

Keep the target separate from the predictor list. Exclude applicant ID
and the original `default_12m` outcome from model inputs.

In [2]:
# Work on a copy; leave the monthly source data unchanged.
monthly = panel.sort_values(["applicant_id", "month_index"]).copy()
monthly["delinquent_month"] = monthly["days_past_due"].gt(0).astype(int)

# Full observation-window summaries.
behaviour_features = monthly.groupby("applicant_id").agg(
    max_dpd_24m=("days_past_due", "max"),
    delinquent_months_24m=("delinquent_month", "sum"),
    overdrafts_24m=("overdraft_count", "sum"),
    utilization_std_24m=("utilization_onbook", "std"),
)

# Recent three months: month indices 21–23.
recent = (
    monthly.loc[monthly["month_index"].between(21, 23)]
    .groupby("applicant_id")
    .agg(
        utilization_recent_3m=("utilization_onbook", "mean"),
        deposit_recent_3m=("deposit_inflow", "mean"),
        max_dpd_recent_3m=("days_past_due", "max"),
    )
)

# Compare the first six months with the last six months.
early = (
    monthly.loc[monthly["month_index"].between(0, 5)]
    .groupby("applicant_id")
    .agg(
        utilization_first_6m=("utilization_onbook", "mean"),
        deposit_first_6m=("deposit_inflow", "mean"),
    )
)

late = (
    monthly.loc[monthly["month_index"].between(18, 23)]
    .groupby("applicant_id")
    .agg(
        utilization_last_6m=("utilization_onbook", "mean"),
        deposit_last_6m=("deposit_inflow", "mean"),
        overdrafts_recent_6m=("overdraft_count", "sum"),
    )
)

behaviour_features = behaviour_features.join(
    [recent, early, late],
    validate="one_to_one",
)

behaviour_features["utilization_change"] = (
    behaviour_features["utilization_last_6m"]
    - behaviour_features["utilization_first_6m"]
)

assert behaviour_features["deposit_first_6m"].gt(0).all(), (
    "A zero starting deposit requires a separate treatment."
)

behaviour_features["deposit_decline_pct"] = (
    1
    - behaviour_features["deposit_last_6m"]
    / behaviour_features["deposit_first_6m"]
)

# Explicitly choose the predictors; do not include every available column.
FEATURES = [
    "utilization_recent_3m",
    "utilization_change",
    "utilization_std_24m",
    "deposit_recent_3m",
    "deposit_decline_pct",
    "max_dpd_24m",
    "max_dpd_recent_3m",
    "delinquent_months_24m",
    "overdrafts_24m",
    "overdrafts_recent_6m",
    "application_leverage",
    "application_dscr",
    "application_bureau_score",
]

background = onbook[
    ["applicant_id", "leverage", "dscr", "bureau_score", TARGET]
].rename(columns={
    "leverage": "application_leverage",
    "dscr": "application_dscr",
    "bureau_score": "application_bureau_score",
})

ews_data = background.merge(
    behaviour_features.reset_index(),
    on="applicant_id",
    how="left",
    validate="one_to_one",
)

assert len(ews_data) == len(onbook)
assert ews_data["applicant_id"].is_unique
assert np.isfinite(ews_data[FEATURES].to_numpy(dtype=float)).all()
assert TARGET not in FEATURES
assert "default_12m" not in FEATURES
assert "applicant_id" not in FEATURES

print(f"Account-level rows: {len(ews_data):,}")
print(f"Model predictors: {len(FEATURES)}")
print("Feature checks passed.")

display(
    ews_data[
        ["applicant_id", "utilization_recent_3m",
         "utilization_change", "deposit_decline_pct",
         "max_dpd_recent_3m", TARGET]
    ].head().round(4)
)

Account-level rows: 9,887
Model predictors: 13
Feature checks passed.


,applicant_id,utilization_recent_3m,utilization_change,deposit_decline_pct,max_dpd_recent_3m,deterioration_next_6_12mo
0,SME000001,0.1105,0.0984,-0.0707,0,0
1,SME000003,0.0903,-0.1998,-0.2107,0,0
2,SME000005,0.6749,0.3998,-0.1209,0,0
3,SME000007,0.4350,-0.1910,0.0446,0,1
4,SME000008,0.4928,0.1775,0.0128,0,0


## 3. Split Accounts into Development and Test Samples

Use 70% of accounts to develop the early warning model and hold aside
30% to evaluate it.

Split at account level so the same borrower cannot appear in both samples.
Stratify by the deterioration target to preserve a similar event rate.

This tests performance on unseen accounts from the same synthetic population.
It is not an out-of-time test on a later calendar period.

In [3]:
from sklearn.model_selection import train_test_split

RANDOM_SEED = 2026

ews_dev, ews_test = train_test_split(
    ews_data,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=ews_data[TARGET],
)

ews_dev = ews_dev.reset_index(drop=True)
ews_test = ews_test.reset_index(drop=True)

X_ews_dev = ews_dev[FEATURES].copy()
y_ews_dev = ews_dev[TARGET].astype(int).copy()

X_ews_test = ews_test[FEATURES].copy()
y_ews_test = ews_test[TARGET].astype(int).copy()

overlap = set(ews_dev["applicant_id"]) & set(ews_test["applicant_id"])

assert not overlap, "An account appears in both samples."
assert len(ews_dev) + len(ews_test) == len(ews_data)
assert X_ews_dev.columns.equals(X_ews_test.columns)

split_summary = pd.DataFrame({
    "sample": ["Development", "Test"],
    "accounts": [len(ews_dev), len(ews_test)],
    "deterioration_cases": [y_ews_dev.sum(), y_ews_test.sum()],
    "deterioration_rate": [y_ews_dev.mean(), y_ews_test.mean()],
})

print(f"Account overlap: {len(overlap)}")

display(split_summary.style.format({
    "accounts": "{:,.0f}",
    "deterioration_cases": "{:,.0f}",
    "deterioration_rate": "{:.2%}",
}).hide(axis="index"))

Account overlap: 0


sample,accounts,deterioration_cases,deterioration_rate
Development,"6,920","1,371",19.81%
Test,"2,967",588,19.82%


## 4. Train the Early Warning Model

Fit a LightGBM classifier using the 13 selected predictors and the
simulated deterioration target.

The model estimates the probability of deterioration, which is different
from the application scorecard's probability of default.

Only development accounts are used for fitting. The test accounts remain
held aside for evaluation. We use fixed model settings and do not adjust
them based on test results.

In [5]:
%pip install lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 5.5 MB/s  0:00:00m0:00:0100:01

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
from lightgbm import LGBMClassifier

ews_model = LGBMClassifier(
    objective="binary",
    n_estimators=200,
    learning_rate=0.03,
    num_leaves=7,
    max_depth=3,
    min_child_samples=100,
    reg_lambda=5.0,
    random_state=RANDOM_SEED,
    n_jobs=2,
    verbosity=-1,
)

ews_model.fit(X_ews_dev, y_ews_dev)

# Position of target class 1 in the probability output.
positive_class_index = list(ews_model.classes_).index(1)

ews_prob_dev = ews_model.predict_proba(X_ews_dev)[:, positive_class_index]
ews_prob_test = ews_model.predict_proba(X_ews_test)[:, positive_class_index]

for probabilities in [ews_prob_dev, ews_prob_test]:
    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0) & (probabilities <= 1)).all()

print("Early warning model fitted.")
print(f"Development accounts used for fitting: {len(X_ews_dev):,}")
print(f"Test accounts scored: {len(X_ews_test):,}")

Early warning model fitted.
Development accounts used for fitting: 6,920
Test accounts scored: 2,967


## 5. Evaluate the Model and Review Workload

Report development and held-out test performance:

- ROC-AUC: ability to rank deteriorating accounts above other accounts.
- Average precision: precision-recall ranking performance, compared with
  the sample's deterioration rate as the random-ranking baseline.
- Top-10% capture: share of all deterioration cases found among the
  highest-risk 10% of accounts.
- Lift: top-group deterioration rate divided by the overall deterioration rate.

We assume capacity to review approximately 10% of accounts. Rank by predicted
risk, using applicant ID to break ties consistently.

Use test results to assess performance. Development results show how much
performance changes on accounts used for training. These metrics evaluate
synthetic outcomes; they do not measure the benefit of contacting borrowers.

In [7]:
from sklearn.metrics import roc_auc_score, average_precision_score


def evaluate_ews(sample_name, accounts, probabilities):
    ranked = accounts[["applicant_id", TARGET]].copy()
    ranked["predicted_deterioration"] = np.asarray(probabilities)

    ranked = ranked.sort_values(
        ["predicted_deterioration", "applicant_id"],
        ascending=[False, True],
    )

    review_count = int(np.ceil(len(ranked) * 0.10))
    reviewed = ranked.head(review_count)

    total_cases = int(ranked[TARGET].sum())
    captured_cases = int(reviewed[TARGET].sum())
    base_rate = ranked[TARGET].mean()
    review_share = review_count / len(ranked)
    capture = captured_cases / total_cases

    return {
        "sample": sample_name,
        "roc_auc": roc_auc_score(accounts[TARGET], probabilities),
        "average_precision": average_precision_score(
            accounts[TARGET], probabilities
        ),
        "base_rate": base_rate,
        "reviewed_accounts": review_count,
        "captured_cases": captured_cases,
        "total_cases": total_cases,
        "top_10pct_capture": capture,
        "top_10pct_lift": capture / review_share,
    }


ews_performance = pd.DataFrame([
    evaluate_ews("Development", ews_dev, ews_prob_dev),
    evaluate_ews("Test", ews_test, ews_prob_test),
]).set_index("sample")

display(ews_performance.style.format({
    "roc_auc": "{:.3f}",
    "average_precision": "{:.3f}",
    "base_rate": "{:.2%}",
    "reviewed_accounts": "{:,.0f}",
    "captured_cases": "{:,.0f}",
    "total_cases": "{:,.0f}",
    "top_10pct_capture": "{:.2%}",
    "top_10pct_lift": "{:.2f}x",
}))

test_result = ews_performance.loc["Test"]

print(
    f"Reviewing the highest-risk {int(test_result['reviewed_accounts']):,} "
    f"test accounts captures {int(test_result['captured_cases']):,} "
    f"of {int(test_result['total_cases']):,} deterioration cases."
)

,roc_auc,average_precision,base_rate,reviewed_accounts,captured_cases,total_cases,top_10pct_capture,top_10pct_lift
sample,,,,,,,,
Development,0.739,0.459,19.81%,692,388,"1,371",28.30%,2.83x
Test,0.700,0.379,19.82%,297,135,588,22.96%,2.29x


Reviewing the highest-risk 297 test accounts captures 135 of 588 deterioration cases.


## 6. Create the Held-Out Watchlist with Warning Triggers

Rank test accounts by predicted deterioration risk and select the
highest-risk 10% for review.

Attach descriptive warning flags using fixed illustrative thresholds:
- Average utilization above 90% during the last three months.
- Utilization rising by more than 15 percentage points.
- Deposit inflow falling by more than 30%.
- Any missed payment during the last three months.
- At least three overdrafts during the last six months.

These flags describe account behaviour. They are not model explanations
or proven causes of deterioration, and they do not change the ranking.
The target outcome is excluded from the displayed review queue.

In [8]:
# Retain the target in ews_test for evaluation, not in the review queue.
test_scored = ews_test[["applicant_id"] + FEATURES].copy()
test_scored["predicted_deterioration"] = ews_prob_test

trigger_flags = pd.DataFrame({
    "High utilization": test_scored["utilization_recent_3m"].gt(0.90),
    "Rising utilization": test_scored["utilization_change"].gt(0.15),
    "Deposit decline": test_scored["deposit_decline_pct"].gt(0.30),
    "Recent missed payment": test_scored["max_dpd_recent_3m"].gt(0),
    "Frequent overdrafts": test_scored["overdrafts_recent_6m"].ge(3),
})

test_scored["warning_reasons"] = trigger_flags.apply(
    lambda row: "; ".join(row.index[row])
    or "Model-ranked risk; no listed trigger",
    axis=1,
)

test_scored = test_scored.sort_values(
    ["predicted_deterioration", "applicant_id"],
    ascending=[False, True],
).reset_index(drop=True)

test_scored.insert(0, "review_rank", np.arange(1, len(test_scored) + 1))

review_count = int(np.ceil(len(test_scored) * 0.10))
test_watchlist = test_scored.head(review_count).copy()

assert TARGET not in test_watchlist.columns
assert len(test_watchlist) == 297

print(f"Held-out accounts prioritized for review: {len(test_watchlist):,}")

display(
    test_watchlist[
        ["review_rank", "applicant_id", "predicted_deterioration",
         "utilization_recent_3m", "deposit_decline_pct",
         "max_dpd_recent_3m", "warning_reasons"]
    ].head(10).style.format({
        "predicted_deterioration": "{:.2%}",
        "utilization_recent_3m": "{:.2%}",
        "deposit_decline_pct": "{:.2%}",
        "max_dpd_recent_3m": "{:.0f}",
    }).hide(axis="index")
)

Held-out accounts prioritized for review: 297


review_rank,applicant_id,predicted_deterioration,utilization_recent_3m,deposit_decline_pct,max_dpd_recent_3m,warning_reasons
1,SME003868,78.36%,123.93%,39.26%,0,High utilization; Rising utilization; Deposit decline; Frequent overdrafts
2,SME010564,76.91%,112.69%,17.51%,0,High utilization; Frequent overdrafts
3,SME005489,75.85%,118.93%,30.73%,0,High utilization; Rising utilization; Deposit decline; Frequent overdrafts
4,SME003202,72.70%,122.82%,18.77%,0,High utilization; Rising utilization; Frequent overdrafts
5,SME002367,71.75%,83.38%,8.87%,0,Frequent overdrafts
6,SME006919,69.56%,118.67%,25.65%,0,High utilization; Rising utilization; Frequent overdrafts
7,SME008428,69.02%,117.20%,13.93%,0,High utilization; Rising utilization; Frequent overdrafts
8,SME008975,68.54%,115.70%,25.33%,0,High utilization; Rising utilization; Frequent overdrafts
9,SME010950,67.78%,86.16%,19.42%,0,Rising utilization; Frequent overdrafts
10,SME007302,66.29%,105.46%,4.47%,30,High utilization; Rising utilization; Recent missed payment


## 7. Save and Verify the Watchlist Outputs

Save the fitted model, ordered predictor list, target definition,
warning-trigger thresholds, and review-capacity assumption.

Export:
- Development and test performance.
- The held-out test watchlist.
- All test-account scores.
- A separate test evaluation file containing the synthetic outcome.

Reload the saved model and verify that its test predictions match.
The review queue excludes the outcome label.

In [9]:
import joblib
import lightgbm
from datetime import datetime, timezone

model_directory = root / "models" / "sme_ews_v1"
report_directory = root / "reports" / "sme_ews_v1"

model_directory.mkdir(parents=True, exist_ok=True)
report_directory.mkdir(parents=True, exist_ok=True)

ews_bundle = {
    "model": ews_model,
    "features": FEATURES.copy(),
    "target": TARGET,
    "positive_class": 1,
    "review_share": 0.10,
    "ranking": "Probability descending; applicant_id ascending for ties",
    "trigger_thresholds": {
        "utilization_recent_3m_above": 0.90,
        "utilization_change_above": 0.15,
        "deposit_decline_pct_above": 0.30,
        "max_dpd_recent_3m_above": 0,
        "overdrafts_recent_6m_at_least": 3,
    },
    "metadata": {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "synthetic_data": True,
        "observation_month_indices": [0, 23],
        "recent_3m_indices": [21, 23],
        "early_6m_indices": [0, 5],
        "late_6m_indices": [18, 23],
        "target_note": (
            "Simulated future deterioration label; "
            "not observed outcomes from additional future months."
        ),
        "development_accounts": len(ews_dev),
        "test_accounts": len(ews_test),
        "random_seed": RANDOM_SEED,
        "lightgbm_version": lightgbm.__version__,
        "limitation": (
            "Held-out accounts from the same synthetic population; "
            "not an out-of-time or intervention-effect evaluation."
        ),
    },
}

bundle_path = model_directory / "ews_bundle.joblib"
joblib.dump(ews_bundle, bundle_path)

# Verify the saved model reproduces the predictions.
loaded_ews_bundle = joblib.load(bundle_path)
loaded_model = loaded_ews_bundle["model"]
class_position = list(loaded_model.classes_).index(
    loaded_ews_bundle["positive_class"]
)

reloaded_probabilities = loaded_model.predict_proba(
    ews_test[loaded_ews_bundle["features"]]
)[:, class_position]

assert np.allclose(
    reloaded_probabilities, ews_prob_test,
    atol=1e-12, rtol=1e-10,
), "Reloaded predictions do not match."

# Review outputs: no outcome labels.
ews_performance.to_csv(report_directory / "performance.csv")
test_watchlist.to_csv(report_directory / "test_watchlist.csv", index=False)
test_scored.to_csv(report_directory / "test_scored.csv", index=False)

# Evaluation output: attach outcomes by ID, not row position.
test_evaluation = test_scored.merge(
    ews_test[["applicant_id", TARGET]],
    on="applicant_id",
    how="left",
    validate="one_to_one",
)

assert test_evaluation[TARGET].notna().all()
test_evaluation.to_csv(
    report_directory / "test_evaluation.csv", index=False
)

print("Model save/reload verification passed.")
print(f"Model saved: {bundle_path.relative_to(root)}")
print(f"Reports saved: {report_directory.relative_to(root)}")
print(f"Held-out watchlist exported: {len(test_watchlist):,} accounts")

Model save/reload verification passed.
Model saved: models/sme_ews_v1/ews_bundle.joblib
Reports saved: reports/sme_ews_v1
Held-out watchlist exported: 297 accounts
